<font size="+4">`Signal and Audio Processing`</font>

<font size="+3">`Seminar 09: Intro to Generative Text-to-Speech`</font>

<font size="+2">`Maks Nakhodnov & Dmitry Kropotov`</font>

<font size="+2">`Bremen, 2026`</font>

What you will learn from this notebook:

* **TTS task formulation:** The *One-to-Many* problem. Evaluation of speech generation.

# `1. TTS Task Formulation and the One-to-Many Problem`

**Text-to-Speech (TTS)** is the task of generating human speech from text. While ASR is a *recognition* task (compressing a continuous noisy signal into discrete semantic tokens), TTS is a **generation** task requiring the restoration of lost information.

Let $Y = (y_1, y_2, \dots, y_N)$ be the input text sequence, where $y_n \in V$ (letters, phonemes, or tokens). 
The TTS task is to generate an audio signal $S = (\mathbf{s}_1, \dots, \mathbf{s}_T)$ or a sequence of acoustic features $X = (\mathbf{x}_1, \dots, \mathbf{x}_M)$ by maximizing the probability:

$$ X^* = \arg\max_{X} P(X \mid Y, \mathcal{C}) $$

Where $\mathcal{C}$ is a contextual condition (speaker identity, emotion, speed, room acoustics).

## `1.1. The One-to-Many Problem`

The main fundamental difficulty in TTS is that the mapping from text to audio is **not injective**. The same phrase "Hello, how are you?" can be pronounced in an infinite number of ways:
* **Pitch, Timbre:** With different timbres (man, woman, child).
* **Prosody, Speed:** With different speeds and pauses.
* **Energy, Emotions:** With different intonations (sarcasm, joy, sadness).
* **Environment:** In different environments, with different background noise.

<b style='color:red;'>Why can't we simply compute the L1/MSE Loss between the predicted and reference spectrograms in TTS, as is done, for example, in image generation?</b>

<details><summary>Answer:</summary>>> 
    
1. Time alignment problem: Speech generated slightly faster or slower than the reference will yield a large MSE loss, even if it sounds perfect to a human. <br>
2. MSE penalizes the model for choosing a different, yet valid intonation. The model learns to predict the expected value of all spectrograms for a given text, which physically looks like a blurred spectrogram without clear harmonics. This *Oversmoothing* effect leads to the generation of a dull, robotic voice.
</details>

## `1.2. TTS Pipeline Structure`

<img src="ASR_vs_TTS.svg" alt="" >

There are many options for choosing each component in the TTS pipeline:

1. **Pre-processing:**
    * Text $\to$ Normalization $\to$ Phonemes $\to$ Prosody extraction.
    * Text $\to$ BPE tokens \& Audio reference $\to$ Discrete audio tokens.
    * ...
2. **Acoustic Model:**
    * Phonemes $\to$ Mel-spectrogram (*Tacotron 2*, *FastSpeech 2* models).
    * Joint modeling of semantics and acoustics (LLMs or Flow models).
    * ...
3. **Acoustic Features:**
    * Mel/Linear-spectrogram
    * Latent hidden representations
    * Discrete audio codes
    * ...
5. **Vocoder:**
    * Mel-spectrogram $\to$ Waveform (*HiFi-GAN*, *Vocos* models).
    * Decoding tokens or latents back into raw audio.
    * ...

Regardless of the intermediate representation (spectrogram or tokens), the final step is always the conversion into a raw sound wave — a one-dimensional array of amplitudes with a sampling rate of $24\,000 - 48\,000$ samples per second. This is handled by the **Vocoder**.

<b style='color:red;'>Why can't we discard the vocoder and force the acoustic model to predict the Raw Waveform directly (e.g., autoregressively)?</b>

<details><summary>Answer:</summary>>> 
Because of the sampling rate. For 1 second of audio at 24 kHz, the model needs to perform 24,000 autoregressive steps. WaveNet did exactly this, but its inference took minutes for a one-second fragment. Vocoders solve the "resolution" problem: the acoustic model operates at a low frequency (\~50 Hz), predicting meaning and melody, while the vocoder generates high-frequency acoustic details up to 24 kHz in parallel.
</details>

## `1.3. Text Frontend: Normalization and G2P`

Before the acoustic model starts generating sound, the raw text must be prepared. Although modern LLM-based approaches can use language structure information for E2E text preprocessing, high-quality preprocessing remains crucial for real-world systems.

### `Text Normalization`

TTS models cannot read numbers, symbols, and abbreviations directly. Normalization converts non-alphabetic characters into their spoken (orthographic) representation:

| Raw text | Context | Normalized text (how it should be read) |
| :--- | :--- | :--- |
| `1984` | In **1984** a novel was released... | *nineteen eighty-four* |
| `1984` | My PIN code: **1984** | *one nine eight four* |
| `St.` | **St.** Patrick's Day | *Saint* |
| `St.` | Wall **St.** | *Street* |
| `$120.5` | Price: **$120.5** | *one hundred twenty dollars and fifty cents* |

**It is important to note that this task is non-trivial, as the result of disambiguation depends on the context!.**

Historically, sets of rules manually written by linguists were used to solve this task. Today, neural models or LLMs are applied, operating in a raw-to-normalized text translation mode.

<b style='color:red;'>What is the danger of using neural networks for text normalization?</b>

<details><summary>Answer:</summary>>> 
    
<b>Hallucinations and word omissions.</b> If manual rules do not know a word, it will simply be spelled out. An LLM, however, might rephrase the text, discard an "inconvenient" digit, or replace it with a random one. Therefore, hybrid Rule-based + Neural approaches still prevail in the industry.

</details>

### `Grapheme-to-Phoneme (G2P)`

Letters (graphemes) do not always unambiguously reflect the sound of a word. In English, the same combination of letters can be read in dozens of ways (e.g., *read* $\to$ /riːd/ or /rɛd/). In Russian, there is vowel reduction (корова $\to$ [к а р о́ в а]) and **homographs** (words that are spelled the same but sound differently due to stress):

* I opened the door **lock** (*замóк*).
* We went on an excursion to the old **castle** (*зáмок*).

G2P models place stresses and convert normalized text into a transcription.

<b style='color:red;'>Why convert text into phonemes at all, if modern Audio LLMs can work directly with text BPE tokens?</b>

<details><summary>Answer:</summary>>> 
    
Using BPE indeed simplifies the pipeline, and models learn to implicitly associate tokens with sounds. However:
1. BPE splits words logically, not phonetically. Rare words, proper names, or new loanwords will be read by the model with stress or pronunciation errors.
2. Phonemes make the model more stable and allow for manual correction of pronunciation errors in dictionaries without retraining the entire acoustic model. Nevertheless, SOTA models increasingly tend to abandon explicit G2P when scaling data.

</details>

More discussion on this topic [here](https://huggingface.co/blog/hexgrad/g2p).

## `1.4. Evaluation Methods and Metrics`

### `Performance Metrics`

Evaluating the speed of a TTS pipeline follows a similar path as for ASR:

1. **RTF (Real-Time Factor):**
  
   This is the ratio of the time spent processing the audio to the duration of the audio itself:

   $$\text{RTF} = \frac{\text{Processing Time}}{\text{Audio Duration}}$$
   For modern Flow Matching models, $\text{RTF} \approx 0.05$.

   Similarly to ASR, the reciprocal value is sometimes considered: $\text{RTFx} = \frac{1}{\text{RTF}}$

2. **First Chunk Latency:**

   Just as the Time to First Token was important for Streaming ASR, for TTS, the time to the first generated chunk is measured.

<img src="https://pic3.zhimg.com/v2-29920fb92ba21fdc5fcdb6270e1e141c_1440w.jpg" alt="" >

### `Objective Quality Metrics`

There is no single mathematical quality metric in the generation task, as "naturalness" is subjective. Historically, acoustic metrics were used, but today the industry has shifted to hybrid and LLM-based approaches.

1. **Word Error Rate / CER:**

   Measures *Intelligibility* and stability. The generated audio is recognized through an ASR pipeline using strong models, typically *Whisper*. If the generative model "swallows" letters, loops, or hallucinates, the WER will increase.

<b style='color:red;'>Why would this approach work? Won't ASR add its own recognition errors?</b>

<details><summary>Answer:</summary>>> 

Typically, TTS generation means generating studio-quality audio. For the domain of high-quality audio without background noise, the error of modern ASR models is minimal.
 
</details>
   
2. **Speaker Similarity (SS):**

   Measures the quality of *Zero-Shot Voice Cloning*. Speaker embeddings from the original reference audio and from the generated audio are extracted using a pre-trained model. 
   $$ \text{SIM} = \cos(\mathbf{e}_{\text{ref}}, \mathbf{e}_{\text{gen}}) = \frac{\mathbf{e}_{\text{ref}} \cdot \mathbf{e}_{\text{gen}}}{\|\mathbf{e}_{\text{ref}}\| \|\mathbf{e}_{\text{gen}}\|} $$
   Values of $\text{SIM} > 0.7$ usually mean that a human will not be able to tell the voices apart.

| Model | Text token | Speech Token | WER (%) | #INS+DEL | #SUB | SS |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Original | - | - | 3.01 | 66 | 200 | 69.67 |
| VALL-E (Wang et al., 2023) | Phone | Encodec | 18.70 | 342 | 1312 | 53.19 |
| UniAudio (Yang et al., 2023) | Phone | Encodec | 8.74 | 254 | 519 | 47.56 |
| SpearTTS (Kharitonov et al., 2023) | Phone | Hubert | 6.14 | 133 | 410 | 51.71 |
| Exp-1-LibriTTS | Phone | Hubert | 7.41 | 325 | 409 | 67.85 |
| Exp-2-LibriTTS | Phone | S<sup>3</sup><sub>en</sub> | 5.05 | 122 | 325 | 67.85 |
| Exp-3-LibriTTS | BPE<sub>en</sub> | S<sup>3</sup><sub>en</sub> | 3.93 | 108 | 239 | 67.85 |
| Exp-4-LibriTTS | BPE | S<sup>3</sup> | 4.76 | 134 | 287 | 65.94 |
| Exp-4-Large-scale | BPE | S<sup>3</sup> | **3.17** | **96** | **184** | **69.49** |

3.[**Fréchet DeepSpeech Distance (FDSD) / Kernel DeepSpeech Distance (KDSD):**](https://arxiv.org/pdf/1909.11646)

   Analogous to FID/KID in CV. It measures the distance between the distributions of real and generated speech in the feature space of a pre-trained ASR.
   $$ \text{FDSD} = \mathcal{W}_{2}(\mathcal{P}_{\text{reference}}, \mathcal{P}_{\text{source}}) \approx \mathcal{W}_{2}(\mathcal{N}(\mu_r, \Sigma_r), \mathcal{N}(\mu_s, \Sigma_s))$$

   $$
    \text{KDSD} = \text{MMD}(f_{\text{reference}}, f_{\text{source}})^{2}
   $$

### `Subjective Quality Metrics`

1. **MOS (Mean Opinion Score):** Human evaluation from 1 to 5 (usually, based on Naturalness and Speaker Similarity parameters). Time-consuming and expensive.

| Model | MOS | FDSD | cFDSD | KDSD ×10<sup>5</sup> | cKDSD ×10<sup>5</sup> |
| :--- | :--- | :--- | :--- | :--- | :--- |
| *natural speech* | 4.55 ± 0.075 | 0.161 | N/A | 0 | 0 |
| *WaveNet*, van den Oord et al. (2016) | 4.41 ± 0.069 | | | | |
| *Parallel WaveNet*, van den Oord et al. (2018) | 4.41 ± 0.078 | | | | |
| FullD | 1.889 ± 0.057 | 4.51 | 4.46 | 785 | 782 |
| cRWD<sub>1</sub> | 3.394 ± 0.058 | 0.362 | 0.247 | 35.2 | 30.9 |
| cRWD<sub>{1,2,4,8,15}</sub> | 3.498 ± 0.059 | 0.398 | 0.284 | 42.1 | 37.9 |
| cRWD<sub>1</sub> + uRWD<sub>1</sub> | 3.502 ± 0.057 | 0.259 | 0.144 | 16.6 | 12.3 |
| (cRWD<sub>1</sub> + uRWD<sub>1</sub>)<sup>×5</sup> | 3.526 ± 0.054 | 0.194 | 0.073 | 5.59 | 1.34 |
| RWD<sub>1,240×{1,2,4,8,15}</sub> | 4.154 ± 0.050 | 0.184 | 0.061 | 3.73 | 0.54 |
| RWD<sup>*</sup><sub>480</sub> | 4.195 ± 0.045 | 0.193 | 0.069 | 5.28 | 0.98 |
| GAN-TTS (RWD<sup>*</sup>) | 4.213 ± 0.046 | 0.184 | 0.060 | 3.84 | 0.37 |

2. **ABX Test:** The listener is given audio A (model 1), audio B (model 2), and a reference X. They must choose which is closer to X.
3.[**MLLM-as-a-Judge**:](https://arxiv.org/pdf/2510.14664v1)

        With the growing popularity of *Instruction-guided TTS* (when the model uses complex instructions, for example: «read this sarcastically»), classical metrics have stopped working. The current standard is to use multimodal LLMs as automatic assessors.
    The generated audio, text, and prompt instruction are loaded into the MLLM, after which it evaluates:
    * **Instruction Following:** Did the model follow the complex instruction (pauses, accent, sighs)?
    * **Expressiveness:** The emotional richness of the speech.

### `Datasets and SOTA Benchmarks`

In ASR, for high-quality and robust audio recognition, the data must be diverse. However, for TTS, the picture is the opposite — if something different from the model's input is spoken in the audio generation, the TTS will learn incorrect pronunciations. Moreover, unconditional models without additional speaker information will average the speech of different speakers, leading to unnatural generation. Therefore, for older TTS models, **clean single-speaker studio datasets** were almost always used. However, modern models that allow **speaker conditioning** (speaker embedding, style tokens, semantic prompts, etc.) are effectively trained on **multi-speaker** datasets. Such models are trained on hundreds of thousands of hours of speech (from podcasts, YouTube) so that the model observes the full variety of timbres and acoustic conditions.

<b style='color:red;'>ASR datasets almost always use a sampling rate of 16 kHz. Why is 24 kHz or even 48 kHz the standard for modern TTS datasets?</b>

<details><summary>Answer:</summary>>> 

1. In ASR, the main goal is to extract meaning. All essential linguistic and semantic information of human speech (formants of vowels and most consonants) lies in the frequency range up to 8 kHz. According to the Nyquist-Shannon sampling theorem, a 16 kHz sampling rate is sufficient to losslessly reconstruct frequencies up to 8 kHz. 
2. In TTS, the goal is naturalness and sound quality. Frequencies above 8 kHz contain higher overtones, breathing, crisp sibilants, and the individual color of the timbre. If a voice is generated at 16 kHz, it will sound muffled. For clean studio sound, the generation of high frequencies is required, which is why TTS datasets and vocoders operate at 24 kHz (covering frequencies up to 12 kHz) and higher.

</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
  <tr style="background-color: #f2f2f2; font-weight: bold; border-bottom: 2px solid black;">
    <th style="padding: 10px;">Dataset Name</th>
    <th style="padding: 10px;">Size (Hours)</th>
    <th style="padding: 10px;">Domain / Features</th>
    <th style="padding: 10px;">Year</th>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LJSpeech</b></td>
    <td style="padding: 10px;">24</td>
    <td style="padding: 10px;">Studio recording, 1 speaker. Historical baseline for Tacotron/FastSpeech.</td>
    <td style="padding: 10px;">2017</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LibriTTS</b></td>
    <td style="padding: 10px;">585</td>
    <td style="padding: 10px;">Audiobooks, 2456 speakers. Standard for early multi-speaker models.</td>
    <td style="padding: 10px;">2019</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>GigaSpeech</b></td>
    <td style="padding: 10px;">10,000</td>
    <td style="padding: 10px;">Podcasts, YouTube. Spontaneous speech, noises. Kickstarted zero-shot models.</td>
    <td style="padding: 10px;">2021</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>Emilia</b></td>
    <td style="padding: 10px;">> 200,000</td>
    <td style="padding: 10px;">Large-scale in-the-wild datasets in different languages. High variability.</td>
    <td style="padding: 10px;">2025</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd">
    <td style="padding: 10px;"><b>SpeechCraft / Parler-TTS</b></td>
    <td style="padding: 10px;">2,000 / 50,000</td>
    <td style="padding: 10px;"><b>Description-based:</b> Audio is accompanied by a text description ("A man speaks calmly...").</td>
    <td style="padding: 10px;">2024</td>
  </tr>
</table>

### `Examples of Metrics for Modern Models`

Evaluation of models in generation and in the model's ability to clone a voice using a 3-second reference on the LibriSpeech test-clean benchmark. More metrics can be [found here](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512#evaluation).

| Model | Architecture | WER $\downarrow$<br>*(Single Speaker / Zero-Shot)* | SS (Voice Similarity) $\uparrow$ | RTF $\downarrow$ |
| :--- | :--- | :---: | :---: | :---: |
| *Ground Truth* | *Human (LibriSpeech)* | *1.8 - 2.1* | *0.730* | *-* |
| [Tacotron 1](https://arxiv.org/abs/1703.10135) (2017) | RNN (Seq2Seq) + Griffin-Lim | \~4.0 / -  | low | \~0.50 - 1.0 |
| [Tacotron 2](https://arxiv.org/abs/1712.05884) (2017) | CNN + BiLSTM + Attention | \~2.5 / **>10.0** | low | > 1.0 (WaveNet)<br>\~0.10 (WaveGlow) |
|[FastSpeech 1](https://arxiv.org/abs/1905.09263) (2019) | Transformer (NAR) | \~3.0 / **>8.0** | low | ~0.03 |
|[FastSpeech 2](https://arxiv.org/abs/2006.04558) (2020) | Transformer (NAR) + Variance Adaptors| \~2.5 / **\~7.7** | low | **\~0.01 - 0.02** |
| [VALL-E](https://arxiv.org/abs/2301.02111) (2023) | AR + NAR | 5.9 | 0.580 | 0.50 |
| [VoiceBox](https://arxiv.org/abs/2306.15687) (2023) | Flow Matching | 1.9 | 0.681 | 0.64 |
| [CosyVoice](https://arxiv.org/abs/2407.05407) (2024)| LLM + Flow Matching | 3.2 | 0.695 | 0.92 |
| [F5-TTS](https://arxiv.org/abs/2410.06885) (2024) | Flow Matching | 2.0 | 0.647 | 0.15 |
|[Seed-TTS](https://arxiv.org/abs/2406.02430) (2024) | AR + DiT | 2.2 | **0.762** | 0.13 |